# Section 09 — System-Wide Control with Plugins
## Lecture 9.4 — Proving Global Coverage: Stress-Testing the AuditPlugin

In the previous lecture you built an `AuditPlugin`, registered it once on the `App`, and ran a single four-turn conversation through the Autonomous Event Genie. The resulting `audit_trail.json` showed every agent that ran during that one conversation.

This notebook answers the natural follow-up question: does that coverage hold up when the orchestrator takes a *different* path through the system? You will run a second, different conversation through the exact same `AuditPlugin` and `App` code, unchanged, and compare the two resulting audit trails side by side.

No new Plugin code is written in this notebook. The `AuditPlugin` class and its registration on the `App` are carried over exactly as they were built in the previous lecture.

## Cell 2 — Install the Google ADK Package

This notebook uses the Google Agent Development Kit (ADK), Google's Python framework for building and running LLM-backed agents. The cell below installs it.

The version is pinned to `google-adk==2.6.3` so this notebook behaves the same way regardless of what has shipped on PyPI since. If the package is already present in this Colab session, the install completes almost instantly and moves on.

In [ ]:
# Pinned for reproducibility. To use the latest version,
# run: pip install google-adk
# Or substitute your preferred version below.
!pip install google-adk==2.6.3 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 23.2 MB/s eta 0:00:00


## Cell 3 — Configure the Google API Key

Every agent in this notebook calls the Gemini API, which needs an API key. This course uses **Google Colab Secrets** exclusively, so the key never appears in plain text in the notebook itself.

**To add the secret in Colab:**
1. Click the key icon (🔑) in the left sidebar of this Colab notebook.
2. Click **Add new secret**.
3. Set the name to `GOOGLE_API_KEY` and paste in a key generated from [Google AI Studio](https://aistudio.google.com/apikey).
4. Toggle **Notebook access** on for this notebook.

**Running locally instead of Colab?** Set `GOOGLE_API_KEY` as an environment variable in your terminal before starting Jupyter, rather than using `userdata.get`.

In [ ]:
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

## Cell 4 — Declare the Model Name

Every agent defined later in this notebook references `MODEL_NAME` instead of a hardcoded model string. Changing the value in this one cell updates the model used by every agent in the notebook at once.

| Variable | Value | Purpose |
|---|---|---|
| `MODEL_NAME` | `"gemini-3.7-flash"` | The model used by every agent defined below |

In [ ]:
# See latest models at: https://ai.google.dev/gemini-api/docs/models
MODEL_NAME = "gemini-3.7-flash"

## Cell 5 — Define the Custom Tools

The Autonomous Event Genie relies on six plain Python functions instead of one monolithic tool: writing to session state, managing the guest list, summing costs, and breaking out of a loop. `GUEST_DATABASE` is a simple in-memory list, and `COMPLETION_PHRASE` is the exact string `accountant_agent` writes when a plan is on budget.

Four of these six functions take a `tool_context` parameter, which is how a plain Python function reaches into session state or, in the case of `exit_loop`, sets the `escalate` action that tells a `LoopAgent` to stop.

In [ ]:
from google.adk.tools import ToolContext

GUEST_DATABASE = []

COMPLETION_PHRASE = "The plan is within the budget."


def add_guest(name: str, email: str) -> dict:
    """Adds a guest to the guest database.

    Args:
        name: The full name of the guest.
        email: The guest's email address.

    Returns:
        A dictionary confirming the guest was added.
    """
    GUEST_DATABASE.append({"name": name, "email": email})
    return {"status": "success", "guest": name}


def get_guest_list(tool_context: ToolContext) -> dict:
    """Retrieves the current guest list and writes it to session state.

    Args:
        tool_context: The tool context, used to write to session state.

    Returns:
        A dictionary containing the current guest list.
    """
    tool_context.state["guest_list"] = GUEST_DATABASE
    return {"guest_list": GUEST_DATABASE}


def sum_costs(costs: list[float]) -> float:
    """Sums a list of costs.

    Args:
        costs: A list of numeric cost values.

    Returns:
        The total of the provided costs.
    """
    return sum(costs)


def exit_loop(tool_context: ToolContext) -> dict:
    """Signals the budget refinement loop to stop iterating.

    Args:
        tool_context: The tool context, used to set the escalate action.

    Returns:
        A dictionary confirming the loop will exit.
    """
    tool_context.actions.escalate = True
    return {"status": "loop_exited"}


def update_session_state(
    tool_context: ToolContext,
    event_type: str,
    city: str,
    budget: float,
) -> dict:
    """Writes the extracted event details to session state.

    Args:
        tool_context: The tool context, used to write to session state.
        event_type: The type of event being planned.
        city: The city the event will be held in.
        budget: The total budget for the event.

    Returns:
        A dictionary confirming the values that were written.
    """
    tool_context.state["event_type"] = event_type
    tool_context.state["city"] = city
    tool_context.state["budget"] = budget
    return {"event_type": event_type, "city": city, "budget": budget}


def send_mock_email(tool_context: ToolContext) -> dict:
    """Simulates sending the drafted announcement email.

    Args:
        tool_context: The tool context, used to read the drafted email.

    Returns:
        A dictionary confirming the simulated send.
    """
    email_draft = tool_context.state.get("email_draft", {})
    recipients = [guest["email"] for guest in GUEST_DATABASE]
    return {
        "status": "sent",
        "subject": email_draft.get("subject", ""),
        "recipient_count": len(recipients),
    }

## Cell 6 — Wrap Google Search as a Sub-Agent Tool

`cost_cutter_agent` needs to both search and call `exit_loop` in the same agent, but ADK will not let you attach the built-in `google_search` tool alongside other tools on one agent. This cell works around that by wrapping a dedicated search agent as an `AgentTool`, which `cost_cutter_agent` can then call side by side with `exit_loop`.

In [ ]:
from google.adk.agents import Agent
from google.adk.tools import google_search
from google.adk.tools.agent_tool import AgentTool

google_search_agent = Agent(
    name="Google_Search_Agent",
    model=MODEL_NAME,
    instruction="You are just a wrapper for the Google Search tool.",
    tools=[google_search]
)
google_search_tool = AgentTool(agent=google_search_agent)

## Cell 7 — Define the Ten Specialist Agents

This is the full specialist layer of the Autonomous Event Genie. `communications_agent` needs a strict output shape, so an `EmailDraft` pydantic model is defined first and passed in as that agent's `output_schema`.

| Agent | Role | Writes to state via `output_key` |
|---|---|---|
| `intake_agent` | Extracts event type, city, budget | (writes via `update_session_state` tool instead) |
| `guest_management_agent` | Manages the guest list | (writes via `get_guest_list` tool instead) |
| `venue_scout_agent` | Finds 3 venues with costs | `venue_options` |
| `catering_scout_agent` | Finds 3 caterers with costs | `catering_options` |
| `entertainment_scout_agent` | Finds 3 entertainment options with costs | `entertainment_options` |
| `initial_plan_synthesizer_agent` | Combines the three scout results into one plan | `current_plan` |
| `accountant_agent` | Picks the cheapest options, checks against budget | `evaluation` |
| `cost_cutter_agent` | Finds cheaper alternatives if over budget | `current_plan` |
| `communications_agent` | Drafts the announcement email | `email_draft` |
| `final_report_agent` | Compiles the final markdown report | (returns final text directly) |

None of these ten agents have any callback attached. That clean slate is exactly what makes the payoff of this lecture visible: the `AuditPlugin` you carry over unchanged later in this notebook covers all ten of them, plus the six agents still to come, without touching a single one of these definitions.

In [ ]:
from pydantic import BaseModel, Field


class EmailDraft(BaseModel):
    subject: str = Field(description="The compelling subject line for the email.")
    body: str = Field(description="The full, well-formatted body of the email.")


intake_agent = Agent(
    name="intake_agent",
    model=MODEL_NAME,
    instruction="From the user's query, identify the event type, city, and budget. Then call update_session_state.",
    tools=[update_session_state]
)

guest_management_agent = Agent(
    name="guest_management_agent",
    model=MODEL_NAME,
    instruction="You are a guest management assistant. Use add_guest and get_guest_list tools.",
    tools=[add_guest, get_guest_list]
)

venue_scout_agent = Agent(
    name="venue_scout_agent",
    model=MODEL_NAME,
    instruction='Find 3 venues for {event_type} in {city} with costs. Output JSON: {"venues": [...]}',
    tools=[google_search],
    output_key="venue_options"
)

catering_scout_agent = Agent(
    name="catering_scout_agent",
    model=MODEL_NAME,
    instruction='Find 3 caterers for {event_type} in {city} with costs. Output JSON: {"caterers": [...]}',
    tools=[google_search],
    output_key="catering_options"
)

entertainment_scout_agent = Agent(
    name="entertainment_scout_agent",
    model=MODEL_NAME,
    instruction='Find 3 entertainment options for {event_type} in {city} with costs. Output JSON: {"entertainment": [...]}',
    tools=[google_search],
    output_key="entertainment_options"
)

initial_plan_synthesizer_agent = Agent(
    name="initial_plan_synthesizer_agent",
    model=MODEL_NAME,
    instruction="Combine {venue_options}, {catering_options}, {entertainment_options} into one JSON with keys venues, caterers, entertainment.",
    output_key="current_plan"
)

accountant_agent = Agent(
    name="accountant_agent",
    model=MODEL_NAME,
    tools=[sum_costs],
    instruction="Select cheapest option from each category in {current_plan}. Use sum_costs. If total > {budget}, output JSON with critique and cheapest_plan. Else output JSON with completion phrase and cheapest_plan.",
    output_key="evaluation"
)

cost_cutter_agent = Agent(
    name="cost_cutter_agent",
    model=MODEL_NAME,
    tools=[google_search_tool, exit_loop],
    instruction="Check {evaluation} critique. If completion phrase, call exit_loop. Else find cheaper alternative for the flagged item. Output updated plan JSON.",
    output_key="current_plan"
)

communications_agent = Agent(
    name="communications_agent",
    model=MODEL_NAME,
    instruction='Draft announcement email from {current_plan}. Output raw JSON only: {"subject": ..., "body": ...}',
    output_key="email_draft",
    output_schema=EmailDraft
)

final_report_agent = Agent(
    name="final_report_agent",
    model=MODEL_NAME,
    instruction="Compile final event plan markdown report using {current_plan}, {venue_options}, {catering_options}, {entertainment_options}, {budget}. Include executive summary, approved plan, alternatives, next steps."
)

## Cell 8 — Define the Five Workflow Agents

These five workflow agents wire the ten specialists together into a single pipeline. `ParallelAgent`, `SequentialAgent`, and `LoopAgent` are ADK's built-in workflow primitives.

| Workflow agent | Type | Role |
|---|---|---|
| `parallel_logistics_scout` | `ParallelAgent` | Runs the three scout agents simultaneously |
| `initial_planning_workflow` | `SequentialAgent` | Scouts, then the synthesizer |
| `budget_refinement_loop` | `LoopAgent` | Alternates `accountant_agent` and `cost_cutter_agent` until on-budget or 3 iterations |
| `budget_optimizer_workflow` | `SequentialAgent` | Wraps the loop |
| `full_plan_workflow` | `SequentialAgent` | Intake, then planning, then budget, then the final report |

**A note on deprecation warnings:** ADK 2.6 introduced a new `Workflow` graph API as the eventual replacement for `SequentialAgent`, `ParallelAgent`, and `LoopAgent`. The ADK documentation currently states that `Workflow` cannot yet be used as an `LlmAgent` sub-agent, which is exactly how `master_orchestrator_agent` consumes these workflow agents via `AgentTool` in the next cell. Migrating now would mean restructuring the orchestrator beyond the scope of this section, so any deprecation warning these classes raise is safe to ignore for now.

In [ ]:
from google.adk.agents import ParallelAgent, SequentialAgent, LoopAgent

parallel_logistics_scout = ParallelAgent(
    name="parallel_logistics_scout",
    sub_agents=[venue_scout_agent, catering_scout_agent, entertainment_scout_agent]
)

initial_planning_workflow = SequentialAgent(
    name="initial_planning_workflow",
    sub_agents=[parallel_logistics_scout, initial_plan_synthesizer_agent]
)

budget_refinement_loop = LoopAgent(
    name="budget_refinement_loop",
    sub_agents=[accountant_agent, cost_cutter_agent],
    max_iterations=3
)

budget_optimizer_workflow = SequentialAgent(
    name="budget_optimizer_workflow",
    sub_agents=[budget_refinement_loop]
)

full_plan_workflow = SequentialAgent(
    name="full_plan_workflow",
    sub_agents=[
        intake_agent,
        initial_planning_workflow,
        budget_optimizer_workflow,
        final_report_agent
    ]
)

/tmp/ipykernel_1194/2137418663.py:3: DeprecationWarning: ParallelAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  parallel_logistics_scout = ParallelAgent(
/tmp/ipykernel_1194/2137418663.py:8: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  initial_planning_workflow = SequentialAgent(
/tmp/ipykernel_1194/2137418663.py:13: DeprecationWarning: LoopAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  budget_refinement_loop = LoopAgent(
/tmp/ipykernel_1194/2137418663.py:19: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  budget_optimizer_workflow = SequentialAgent(
/tmp/ipykernel_119

## Cell 9 — Define the Master Orchestrator

`master_orchestrator_agent` is the single agent a user actually talks to. It never plans, sources venues, or drafts email itself. It reads what the user is asking for and routes the request to one of four tools: the full planning workflow, the guest list manager, the communications drafter, or the send-email tool.

In [ ]:
full_plan_workflow_tool = AgentTool(agent=full_plan_workflow)
guest_list_manager_tool = AgentTool(agent=guest_management_agent)
communications_tool = AgentTool(agent=communications_agent)

master_orchestrator_agent = Agent(
    name="master_orchestrator_agent",
    model=MODEL_NAME,
    instruction="Delegate: full_plan_workflow_tool for planning, guest_list_manager_tool for guests, communications_tool for email drafting, send_mock_email for sending.",
    tools=[
        full_plan_workflow_tool,
        guest_list_manager_tool,
        communications_tool,
        send_mock_email
    ]
)

## Cell 10 — The AuditPlugin (Carried Over Unchanged)

This is the exact same `AuditPlugin` class built in the previous lecture. As a quick reminder: it subclasses `BasePlugin` and overrides four hooks. `before_agent_callback` and `after_agent_callback` log every agent start and end. `before_tool_callback` logs every tool call. `after_run_callback` fires once, after the entire run completes, and flushes the accumulated log to `audit_trail.json`.

Nothing in this class changes in this lecture. It is reproduced here only so this notebook is self-contained.

In [ ]:
import json
from datetime import datetime
from google.adk.plugins import BasePlugin


class AuditPlugin(BasePlugin):
    """
    A system-wide audit plugin that logs every agent invocation,
    tool call, and run completion across all agents in the App.
    Registered once on the App, it covers all 16 agents automatically.
    """

    def __init__(self):
        super().__init__(name="audit_plugin")
        self.log = []

    async def before_agent_callback(self, *, agent, callback_context):
        self.log.append({
            "timestamp": datetime.now().isoformat(),
            "event": "agent_start",
            "agent": agent.name,
        })
        return None

    async def after_agent_callback(self, *, agent, callback_context):
        self.log.append({
            "timestamp": datetime.now().isoformat(),
            "event": "agent_end",
            "agent": agent.name,
        })
        return None

    async def before_tool_callback(self, *, tool, tool_args, tool_context):
        self.log.append({
            "timestamp": datetime.now().isoformat(),
            "event": "tool_call",
            "tool": tool.name,
            "args": str(tool_args),
        })
        return None

    async def after_run_callback(self, *, invocation_context):
        with open("audit_trail.json", "w") as f:
            json.dump(self.log, f, indent=2)
        print(f"✅ Audit trail saved — {len(self.log)} events logged")

## Cell 11 — Session Service and the Run Helper

An `InMemorySessionService` holds the conversation for the life of this notebook run, and `run_agent_query` wraps `runner.run_async()` inside an `async for` loop, handing back the final text response once `event.is_final_response()` returns `True`.

The helper takes an `app`, not a bare agent. `Runner` receives `app=app` directly, which is the current ADK 2.x pattern. `Runner(plugins=[...])` and the legacy `agent=` / `app_name=` form both still run, but they raise a `DeprecationWarning` and are not what this notebook uses.

In [ ]:
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai.types import Content, Part

session_service = InMemorySessionService()


async def run_agent_query(app, query, session, user_id):
    """Initializes a runner and executes a query for a given App and session."""
    runner = Runner(
        app=app,
        session_service=session_service,
    )
    final_response = None
    async for event in runner.run_async(
        user_id=user_id,
        session_id=session.id,
        new_message=Content(parts=[Part(text=query)], role="user"),
    ):
        if event.is_final_response():
            final_response = event.content.parts[0].text
    return final_response

## Cell 12 — Recap Run: Register the AuditPlugin and Replay the Lecture 9.3 Conversation

Before stress-testing anything, this cell reproduces the Lecture 9.3 baseline exactly: a fresh `AuditPlugin()` instance is registered on a fresh `App`, and the same four-turn conversation from that lecture runs again (a 50-person AI tech meetup in Austin, Texas, with a $4000 budget). This produces a known-good audit trail to compare the stress test against.

`App` binds `master_orchestrator_agent` to application-wide configuration, including plugins. Plugins are registered on the `App`, not the `Runner`; `Runner(plugins=[...])` still runs but raises a `DeprecationWarning`, and this course does not use it. The session is created with `app_name=app.name`, because the session service keys sessions by that name and it must match the name on the `App` object exactly.

Watch for the `✅ Audit trail saved` confirmation message, printed once from inside `after_run_callback` after the whole run completes.

In [ ]:
from google.adk.apps import App

app = App(
    name="autonomous_event_genie",
    root_agent=master_orchestrator_agent,
    plugins=[AuditPlugin()]
)

session = await session_service.create_session(
    app_name=app.name, user_id="student_user"
)

response_1 = await run_agent_query(
    app,
    "I need to plan a 50-person AI tech meetup in Austin, Texas with a budget of $4000. Find some vendors.",
    session,
    "student_user"
)
print(response_1)

response_2 = await run_agent_query(
    app,
    "This looks great. Please add 'Grace Hopper' to the guest list. Her email is grace@example.com.",
    session,
    "student_user"
)
print(response_2)

response_3 = await run_agent_query(
    app,
    "Now, please draft an announcement email based on the final plan.",
    session,
    "student_user"
)
print(response_3)

response_4 = await run_agent_query(
    app,
    "Perfect. Please send the email to the guest list.",
    session,
    "student_user"
)
print(response_4)

/usr/local/lib/python3.13/dist-packages/google/adk/models/llm_request.py:273: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  declaration = tool._get_declaration()


✅ Audit trail saved — 31 events logged
✅ Audit trail saved — 32 events logged
Here is a comprehensive event plan and curated vendor options for your **50-person AI Tech Meetup in Austin, Texas** within your **$4,000 budget**.

---

# Event Plan: Austin AI Tech Meetup

### **Key Metrics**
* **Target Headcount:** 50 attendees  
* **Total Budget:** $4,000.00  
* **Recommended Package Cost:** ~$2,050.00 – $2,350.00  
* **Remaining Budget Surplus:** $1,650.00 – $1,950.00  

---

## 1. Recommended Primary Package

| Category | Vendor & Details | Estimated Cost |
| :--- | :--- | :--- |
| **Venue** | **Fibercove** *(South Lamar)*<br>• 4-hour rental ($175/hr)<br>• Capacity: 50 attendees<br>• High-speed Google Fiber, built-in 120" projector screen, AV equipment, cafe/bar area, and free parking | **$700.00** |
| **Catering** | **Torchy's Tacos** *(Austin Tex-Mex)*<br>• Build-your-own taco buffet ($16–$22/person)<br>• Famous Green Chile Queso, chips, salsas, drop-off warming setup, and serving war

## Cell 13 — Save the First Audit Trail Under a Distinct Filename

`after_run_callback` just wrote the recap run's events to `audit_trail.json`. Before running a second conversation, this cell copies that file to a distinct name so the second run does not overwrite it. Both files need to exist side by side for the comparison in a later cell.

In [ ]:
import shutil

shutil.copy("audit_trail.json", "audit_trail_run1_baseline.json")

'audit_trail_run1_baseline.json'

## Cell 14 — Stress-Test Run: A Fresh AuditPlugin, App, and a Different Path

This cell runs a second, different four-turn conversation: a 30-person product launch party in San Francisco with a deliberately, almost absurdly tight $50 budget, and a guest-management turn that adds two guests instead of one. A merely tight budget (a few thousand dollars) is not a strong enough stress test on its own, because `accountant_agent`'s judgment of what counts as "on budget" depends on whatever prices `venue_scout_agent`, `catering_scout_agent`, and `entertainment_scout_agent` find, and those can come back cheaper than expected. A $50 budget for a 30-person event in San Francisco is unrealistic under any reasonable pricing, so `accountant_agent` should reliably flag it over budget, and `cost_cutter_agent` should have to keep searching for savings across more than one iteration of `budget_refinement_loop`, rather than the single pass Run 1 needed. Exactly how many of the loop's three allowed iterations actually run is still the model's live judgment call each time, so treat the specific count as something to observe, not something guaranteed in advance.

A **fresh** `AuditPlugin()` instance and a **fresh** `App` are created here, both assigned to `app` again. This is required, not optional. `AuditPlugin.__init__` sets `self.log = []` once, when the instance is created. Reusing the recap run's `AuditPlugin` instance would mean this run's events append to the same in-memory list as the recap run, merging the two audit trails into one instead of keeping them separate. Creating a fresh instance, and a fresh `App` to register it on, gives this run a clean `self.log` to start from.

Watch which agents show up and how many `accountant_agent` / `cost_cutter_agent` events appear this time, and compare that mentally against what you just saw in the recap run. The last two lines are a diagnostic, not part of the four-turn conversation: they read `evaluation` straight out of session state, which is exactly what `accountant_agent` last wrote there. Printing it directly tells you whether the accountant judged the plan on-budget or not, instead of inferring that indirectly from the loop-event count.

In [ ]:
app = App(
    name="autonomous_event_genie",
    root_agent=master_orchestrator_agent,
    plugins=[AuditPlugin()]
)

session = await session_service.create_session(
    app_name=app.name, user_id="student_user"
)

response_1 = await run_agent_query(
    app,
    "I need to plan a 30-person product launch party in San Francisco with a budget of $50. Find some vendors.",
    session,
    "student_user"
)
print(response_1)

response_2 = await run_agent_query(
    app,
    "Please add 'Ada Lovelace' to the guest list, email ada@example.com, and also add 'Alan Turing', email alan@example.com.",
    session,
    "student_user"
)
print(response_2)

response_3 = await run_agent_query(
    app,
    "Now, please draft an announcement email based on the final plan.",
    session,
    "student_user"
)
print(response_3)

response_4 = await run_agent_query(
    app,
    "Perfect. Please send the email to the guest list.",
    session,
    "student_user"
)
print(response_4)

# Diagnostic: read back what accountant_agent last wrote to session
# state, to see directly whether it judged the plan on-budget.
final_session = await session_service.get_session(
    app_name=app.name, user_id="student_user", session_id=session.id
)
print("\naccountant_agent's last evaluation:")
print(final_session.state.get("evaluation"))

✅ Audit trail saved — 27 events logged
✅ Audit trail saved — 39 events logged
✅ Audit trail saved — 40 events logged
Here is an event plan and vendor breakdown tailored for a **30-person product launch party in San Francisco** within a **$50 budget**:

---

## 1. Budget-Optimized Plan ($45.00 Total)

Given the $50 total budget ($1.67 per person), the recommended approach leverages San Francisco’s premier public open spaces combined with DIY light refreshments and founder-led entertainment.

| Category | Selection / Vendor | Details | Estimated Cost |
| :--- | :--- | :--- | :--- |
| **Venue** | **Salesforce Transit Center Rooftop Park** or **343 Sansome Rooftop POPOS** | Free, high-visibility outdoor venues in the downtown/SoMa tech corridor. Ideal for informal product demos and networking. | **$0.00** |
| **Catering** | **DIY Refreshment Spread (Trader Joe's / Local SF Market)** | Light refreshment package: bulk sparkling waters/ciders, seasonal fruit platters, gourmet cookies, chips, 

## Cell 15 — Compare the Two Audit Trails

This cell loads both saved trails and runs a small `summarize()` function on each. `summarize()` computes several numbers from a raw event log:

| Metric | What it counts | Why it matters here |
|---|---|---|
| Total events | Every entry in the log | Shows overall how much activity a conversation triggered |
| Unique agents ran | Distinct `agent` names seen across the log | Confirms which of the 16 agents this particular path actually touched |
| Tool calls made | Entries where `event == "tool_call"` | A rough proxy for how much work the agents did |
| Budget loop events | `accountant_agent` and `cost_cutter_agent` entries combined | The most meaningful signal to compare across runs, because it directly reflects how many times `budget_refinement_loop` iterated |

The loop-event count is worth watching closely here. Run 1's $4000 Austin budget is realistic enough that `accountant_agent` typically approves the plan on its first pass, so `budget_refinement_loop` runs a single iteration; expect 4 loop events (one `agent_start` and one `agent_end` for each of `accountant_agent` and `cost_cutter_agent`). Run 2's $60 budget for a 30-person San Francisco launch party is unrealistic under any reasonable pricing, so `accountant_agent` should flag it over budget, and `cost_cutter_agent` should have to make at least one real attempt at a cheaper alternative rather than exiting immediately, pushing the loop-event count to 8 or more. The exact number can vary between runs, since it depends on the model's live judgment about when to keep trying versus call `exit_loop`. If your numbers come out differently, or Run 2 also lands on 4, check the diagnostic print at the end of Cell 14 first. It shows exactly what `accountant_agent` last wrote to state, which tells you directly whether it judged the plan on-budget rather than leaving you to infer that indirectly from the loop-event count alone.

In [ ]:
import json

with open("audit_trail_run1_baseline.json") as f:
    run1 = json.load(f)

with open("audit_trail.json") as f:
    run2 = json.load(f)


def summarize(log, label):
    agents = list(dict.fromkeys(e["agent"] for e in log if "agent" in e))
    tool_calls = [e for e in log if e["event"] == "tool_call"]
    loop_agent_events = [
        e for e in log
        if e.get("agent") in ("accountant_agent", "cost_cutter_agent")
    ]
    print(f"\n--- {label} ---")
    print(f"Total events:        {len(log)}")
    print(f"Unique agents ran:   {len(agents)}")
    print(f"Tool calls made:     {len(tool_calls)}")
    print(f"Budget loop events:  {len(loop_agent_events)} (accountant_agent + cost_cutter_agent combined)")


summarize(run1, "Run 1 — Austin meetup, $4000 budget")
summarize(run2, "Run 2 — SF launch party, $50 budget")


--- Run 1 — Austin meetup, $4000 budget ---
Total events:        46
Unique agents ran:   16
Tool calls made:     8
Budget loop events:  4 (accountant_agent + cost_cutter_agent combined)

--- Run 2 — SF launch party, $50 budget ---
Total events:        55
Unique agents ran:   17
Tool calls made:     11
Budget loop events:  8 (accountant_agent + cost_cutter_agent combined)


## Cell 16 — The Coverage Guarantee

Two different conversations. Two different budgets. Two different guest-list turns. A different number of `budget_refinement_loop` iterations in each. And yet both audit trails came from the exact same `AuditPlugin` class and the exact same four-hook design, without a single edit between the two runs.

No matter which path the orchestrator takes, no matter how many loop iterations happen, no matter which agents get invoked more or less, the `AuditPlugin` sees everything, because it is registered on the `App`, not on any individual agent. You did not write a single line of new Plugin code for the second run. You did not touch the `App` registration pattern. The coverage came for free.